# High-Speed Packaging Label Inspection Engine: Quickstart Demo

This interactive notebook demonstrates the **Region-Aware Multi-Stream Machine Vision Engine** on registered physical label photographs.

### Engine Capabilities:
* **Sub-pixel Homography Registration** via 4-fiducial corners.
* **Multi-Stream Inspection**: Margin Paper Spotting, Barcode Frequency Modulation, Text Mass Integrals, and Perimeter Edge Tracing.
* **Deterministic Performance**: 100% sensitivity on missing prints, smudges, and tears in ~80ms on CPU.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from label_inspection import inspect_label, build_empirical_template, ZONES

print('Label Inspection Engine loaded successfully!')
print('Active Semantic Zones:', list(ZONES.keys()))

## 1. Derive Golden Reference Template
The golden template is calculated as the photometric median of pristine normal reference labels.

In [ ]:
data_dir = Path.cwd().parent / 'data' / 'printed_labels'
normal_dir = data_dir / 'Normal'

template = build_empirical_template(train_dir=normal_dir)
print(f'Template shape: {template.shape}, dtype: {template.dtype}')

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(template, cmap='gray')
ax.set_title('Golden Reference Template (Median Normal)')
ax.axis('off')
plt.show()

## 2. Inspect a Missing Print Defect
We evaluate a physical label with a missing barcode under challenging lighting.

In [ ]:
test_dir = data_dir / 'Testing'
sample_path = next(test_dir.glob('M01_*.png'), None)
if sample_path is None:
    sample_path = next(test_dir.glob('*.png'))

print(f'Evaluating sample: {sample_path.name}')
result = inspect_label(sample_path, template=template)

print('-' * 40)
print(f"Decision:   {result['decision']}")
print(f"Score:      {result['score']:.3f} (Threshold tau = 1.0)")
print(f"Trigger:    {result['trigger']}")
print('-' * 40)

# Visualize 4-panel breakdown
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(result['registered'])
axes[0].set_title('1. Registered Capture')
axes[0].axis('off')

axes[1].imshow(result['heatmap'], cmap='magma')
axes[1].set_title('2. Photometric Heatmap')
axes[1].axis('off')

axes[2].imshow(result['mask'], cmap='gray')
axes[2].set_title('3. Detected Anomaly Mask')
axes[2].axis('off')

axes[3].imshow(result['composite'])
axes[3].set_title(f"4. Overlay ({result['trigger']})")
axes[3].axis('off')

plt.tight_layout()
plt.show()